# Model Training for Classification with HydraBLE (Variable Length Finetuning)

In [1]:
import os
from pathlib import Path


os.chdir(Path.cwd().parents[0])
print("Now in:", Path.cwd())

dataPathProcessed = str(Path.cwd()) + r"\data\csv" + r"\Processed Files\\"
checkPointPath = str(Path.cwd()) + r"\out\modeling\checkpoints\classification_finetuning\\"

Now in: C:\Users\stsax\OneDrive\Studium\9. Semester\Masterarbeit\Repository


In [2]:
MAX_TOKEN_LENGTH = 1024
MAX_SEQUENCE_LENGTH = 32

In [3]:
import pickle
import pandas as pd
from modeling.BleDataset import BLEStreamDataset
import torch

from bpe.bpe import BleBytePairEncoder
from modeling.Trainer import TrainerConfig

MaskConfigPath = str(Path.cwd()) + r"\data_masking\mask_configs\\"
picklePath = str(Path.cwd()) + r"\out\pickle_objects\bpe" + "\\"

with open(picklePath + 'Fitted_BPE_State_Dict.pickle', 'rb') as f:
    state_dict = pickle.load(f)


pkt_df_val = pd.read_parquet(dataPathProcessed + r"classification_dataset_v2_test.parquet")
seq_table_val = pd.read_parquet(dataPathProcessed + r"classification_sequence_table_v2_test.parquet")
stream_idx_val = pd.read_parquet(dataPathProcessed + r"classification_stream_index_v2_test.parquet")




dataset_val = BLEStreamDataset(packet_df=pkt_df_val, sequence_table=seq_table_val, stream_index=stream_idx_val,
                           max_sequence_length=MAX_SEQUENCE_LENGTH, max_token_length=MAX_TOKEN_LENGTH,
                           tokenizer_dict=state_dict, mask_config_path=MaskConfigPath, dataset_type='validation', augment_data=True, adapt_sequence_length=True, deterministic=False)

In [4]:
from modeling.HydraBLE import HydraBLETransformer

bpe = BleBytePairEncoder.from_state_dict(state_dict)

model = HydraBLETransformer(
        vocab_size=bpe.vocab_size,
        num_classes=dataset_val.num_of_known_classes,
        pad_id=1,
        max_heads=8,
        head_dim=64,
        depth=4,
        max_len=2048,
        subnet_heads=(1, 2, 4, 8),
        separate_cls_heads=True,
    )


ckpt = torch.load(str(Path.cwd()) + r"\out\modeling\checkpoints\classification_finetuning\\"  + "last.pt", map_location="cuda:0")
model.load_state_dict(ckpt["model_state_dict"])

model.to('cuda:1')
model.eval()

HydraBLETransformer(
  (token_embed): Embedding(2048, 512)
  (blocks): ModuleList(
    (0-3): 4 x HydraEncoderBlock(
      (norm1): HydraLayerNorm()
      (attn): HydraAttention(
        (qkv): HydraLinear()
        (proj): HydraLinear()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (norm2): HydraLayerNorm()
      (mlp): HydraMlp(
        (fc1): HydraLinear()
        (fc2): HydraLinear()
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): HydraLayerNorm()
  (lm_head): HydraLinear()
  (cls_heads): ModuleDict(
    (1): HydraLinear()
    (2): HydraLinear()
    (4): HydraLinear()
    (8): HydraLinear()
  )
)

In [5]:
out = dataset_val[3946]
out

{'tokens': tensor([  4,  12, 244,  ...,   0,   0,   0]),
 'target': tensor([0.0667, 0.0667, 0.0667, 0.0667, 0.0667, 0.0667, 0.0667, 0.0667, 0.0667,
         0.0667, 0.0667, 0.0667, 0.0667, 0.0667, 0.0667]),
 'label': 'Other Device',
 'label_id': 15,
 'masked packets': [(585843, 37, 'D6BE898EC10C417C6FE751C407EC0F25E5ACB2F871'),
  (1230353, 38, 'D6BE898EC10C417C6FE751C407EC0F25E5ACB2F871'),
  (4141520, 39, 'D6BE898EC10C417C6FE751C407EC0F25E5ACB2F871'),
  (3334407, 37, 'D6BE898EC10C417C6FE751C407EC0F25E5ACB2F871'),
  (2008014, 38, 'D6BE898EC10C417C6FE751C407EC0F25E5ACB2F871'),
  (6431057, 39, 'D6BE898EC10C417C6FE751C407EC0F25E5ACB2F871'),
  (612681, 37, 'D6BE898EC10C417C6FE751C407EC0F25E5ACB2F871'),
  (4930021, 37, 'D6BE898EC10C417C6FE751C407EC0F25E5ACB2F871'),
  (3101343, 38, 'D6BE898EC10C417C6FE751C407EC0F25E5ACB2F871'),
  (4606432, 37, 'D6BE898EC10C417C6FE751C407EC0F25E5ACB2F871'),
  (8467452, 39, 'D6BE898EC10C417C6FE751C407EC0F25E5ACB2F871'),
  (1901325, 37, 'D6BE898EC10C417C6FE751C4

In [6]:
out['tokens'][:200]

tensor([  4,  12, 244, 119, 516, 722, 325, 384, 371, 491, 341, 456, 267, 496,
        275, 297, 489, 432, 438, 508, 373,   1,   4,  22, 202,  21, 517, 722,
        325, 384, 371, 491, 341, 456, 267, 496, 275, 297, 489, 432, 438, 508,
        373,   1,   4,  67,  53, 212, 518, 722, 325, 384, 371, 491, 341, 456,
        267, 496, 275, 297, 489, 432, 438, 508, 373,   1,   4,  54, 229,  11,
        516, 722, 325, 384, 371, 491, 341, 456, 267, 496, 275, 297, 489, 432,
        438, 508, 373,   1,   4,  34, 167, 210, 517, 722, 325, 384, 371, 491,
        341, 456, 267, 496, 275, 297, 489, 432, 438, 508, 373,   1,   4, 102,
         37,  85, 518, 722, 325, 384, 371, 491, 341, 456, 267, 496, 275, 297,
        489, 432, 438, 508, 373,   1,   4,  13,  93,  77, 516, 722, 325, 384,
        371, 491, 341, 456, 267, 496, 275, 297, 489, 432, 438, 508, 373,   1,
          4,  79,  61, 233, 516, 722, 325, 384, 371, 491, 341, 456, 267, 496,
        275, 297, 489, 432, 438, 508, 373,   1,   4,  51,  86, 1

In [7]:
from data_processing.LabelLut import LABEL_OTHER_DEVICE

pkt_df_test = pd.read_parquet(dataPathProcessed + r"classification_dataset_v2_test.parquet")

known_labels = sorted(list(pkt_df_test['Label'].unique()))
known_labels.remove(LABEL_OTHER_DEVICE)

label_lut = {i: label for i, label in enumerate(known_labels)}
label_id_unknown = len(label_lut)
label_lut[label_id_unknown] = LABEL_OTHER_DEVICE

In [8]:
import itertools

tokens = bpe.encode_many(out['masked packets'])
tokens = list(itertools.chain(*tokens))
tokens = tokens[:1024]
with torch.no_grad():
    tokens = torch.tensor(tokens).unsqueeze(0).to('cuda:1')

    logits = model(tokens)
    probas = torch.nn.functional.softmax(logits, dim=-1)

    probas = probas.squeeze(0)

    max_proba, _ = torch.max(probas, dim=-1)
    pred = torch.argmax(probas, dim=-1)
    proba_cls = probas[int(pred)]
    pred = label_lut[int(pred)]

    print(pred)
    print(max_proba)

Apple Find My Device (offline)
tensor(0.1041, device='cuda:1')


In [9]:
probas.shape

torch.Size([15])